# 03 — Pipeline de transformation reproductible

**Auteur :** Benoit Girard — CheckItAI  
**Livrable n°3**

Ce notebook déroule l'**étape Transform** : nettoyer, valider, normaliser et exporter les données brutes vers un dataset propre conforme au schéma (`docs/schema_donnees.md`). Le pipeline est organisé en trois temps explicites — **lecture → traitement → export** — et journalisé.

## Définition du *done*

| Critère | Cible |
|---|---|
| Fonctionne sans erreur depuis les données extraites | ✅ |
| Pipeline découpé en étapes claires | ✅ |
| Logs traçant chaque transformation | ✅ |
| Paramètres configurables | ✅ |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from checkitai.logging_setup import setup_logging

setup_logging()

## 1. Les fonctions unitaires de transformation

Chaque transformation est une **petite fonction nommée**, testable indépendamment. Démonstration sur des exemples.

In [2]:
from checkitai.config import TransformConfig
from checkitai.transform import extrait_domaine, nettoie_texte, valide_image

cfg = TransformConfig()
print(repr(nettoie_texte("<p>Bonjour&nbsp;  le   <b>monde</b> !</p>")))
print("image .jpg valide :", valide_image("https://site.com/a.jpg", cfg))
print("page .html rejetée :", valide_image("https://site.com/a.html", cfg))
print("domaine :", extrait_domaine("https://www.bbc.co.uk/news/article"))

'Bonjour le monde !'
image .jpg valide : True
page .html rejetée : False
domaine : bbc.co.uk


## 2. Paramètres configurables

Le comportement du pipeline est piloté par `TransformConfig` (longueur minimale de texte, image obligatoire ou non, format de sortie). On rend ici le mode multimodal **strict** : toute publication sans image valide est écartée.

In [3]:
cfg = TransformConfig(require_image=True, min_text_length=30, output_format="parquet")
cfg

TransformConfig(min_text_length=30, valid_image_extensions=('.jpg', '.jpeg', '.png', '.webp', '.gif'), require_image=True, output_format='parquet')

## 3. Lecture → traitement → export

On applique le pipeline au dernier fichier brut.

In [4]:
from checkitai.config import RAW_DIR
from checkitai.transform import exporte, lit_brut, traite

raw_path = sorted(RAW_DIR.glob("raw_publications_*.json"))[-1]
bruts = lit_brut(raw_path)
df, stats = traite(bruts, cfg)
stats

2026-06-29 18:07:05 | INFO    | checkitai.transform | Transformation : lecture du fichier brut G:\Mon Drive\OC\Projet_12\checkitai\data\raw\raw_publications_20260629_160658.json


2026-06-29 18:07:05 | INFO    | checkitai.transform | Transformation : 142 publications brutes lues


2026-06-29 18:07:05 | INFO    | checkitai.transform | Transformation : 141/142 publications valides apres nettoyage


2026-06-29 18:07:05 | INFO    | checkitai.transform | Transformation : 0 doublons retires


{'total_brut': 142,
 'total_valide': 141,
 'rejetes': 1,
 'doublons': 0,
 'avec_image': 141,
 'labellisees': 24}

Les statistiques montrent l'effet du nettoyage : publications rejetées (texte trop court ou image manquante) et doublons retirés.

In [5]:
df[["source", "title", "has_image", "text_length", "label"]].head(8)

,source,title,has_image,text_length,label
0,rss:the_guardian,‘Everyone is talking about Cape Verde’: World ...,True,567,NaN
1,rss:the_guardian,Whereabouts of nearly 300 people with Ebola un...,True,568,NaN
2,rss:the_guardian,Outrage as woman jailed for three years after ...,True,606,NaN
3,rss:the_guardian,‘Constitutional coup’ claims as Zimbabwe senat...,True,581,NaN
4,rss:the_guardian,France confirms first Ebola case in doctor who...,True,676,NaN
5,rss:the_guardian,Venezuela earthquakes aftershock hits near cap...,True,678,NaN
6,rss:the_guardian,Families of two football players killed and in...,True,695,NaN
7,rss:the_guardian,Weather tracker: North-west US hit by snow ahe...,True,683,NaN


## 4. Contrôle qualité : le lien texte-image

On vérifie l'association texte-image.

In [6]:
assert df["has_image"].all(), "En mode strict, toutes les lignes ont une image"
assert (df["text_length"] >= cfg.min_text_length).all()
print("Contrôles OK : chaque publication a un texte exploitable ET une image.")

Contrôles OK : chaque publication a un texte exploitable ET une image.


In [7]:
chemin = exporte(df, cfg)
print("Dataset exporté :", chemin.name)

2026-06-29 18:07:06 | INFO    | checkitai.transform | Transformation : dataset de 141 lignes exporte vers G:\Mon Drive\OC\Projet_12\checkitai\data\processed\publications_20260629_160705.parquet


Dataset exporté : publications_20260629_160705.parquet


## Conclusion

Le pipeline produit un dataset propre, typé et multimodal, prêt pour le chargement (étape Load) et l'entraînement. Il est reproductible (mêmes entrées → mêmes sorties) et entièrement journalisé.